# HCMST Project — Data Cleaning

## Goals
- load the raw HCMST dataset
- keep selected variables only
- define the active-relationship sample for Wave 1
- clean and recode variables for analysis
- save processed datasets for clustering and modeling

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path("..")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

OUTPUTS = PROJECT_ROOT / "outputs"
OUTPUT_TABLES = OUTPUTS / "tables"
OUTPUT_FIGURES = OUTPUTS / "figures"
OUTPUT_SLIDES = OUTPUTS / "slides"

In [3]:
sav_path = DATA_RAW / "HCMST 2017 to 2022 small public version 2.2.sav"
df = pd.read_spss(sav_path)

print(df.shape)

(3510, 725)


In [4]:
def safe_numeric(series):
    return pd.to_numeric(series.astype(str), errors="coerce")

def recode_yes_no(series):
    s = series.astype(str).str.strip().str.lower()
    return s.map({
        "yes": 1,
        "no": 0
    })

def recode_same_sex(series):
    s = series.astype(str).str.strip()
    return s.map({
        "same_sex_couple": 1,
        "NOT same-sex couple": 0
    })

def recode_quality(series):
    s = series.astype(str).str.strip()
    mapping = {
        "Very Poor": 1,
        "Poor": 2,
        "Fair": 3,
        "Good": 4,
        "Excellent": 5
    }
    return s.map(mapping)

In [5]:
candidate_vars = [
    # identifiers and weights
    "caseid_new",
    "w1_weight_combo",
    "w2_attrition_adj_weights",
    "w3_attrition_adj_weight",
    "w2_surveyed",
    "w3_surveyed",

    # wave 1 demographics
    "w1_ppage",
    "w1_ppeduc",
    "w1_ppincimp",
    "w1_ppgender",
    "w1_ppethm",
    "w1_ppreg9",
    "w1_ppwork",

    # wave 1 relationship structure
    "w1_partnership_status",
    "w1_q19",
    "w1_married",
    "w1_same_sex_couple",
    "w1_relate_duration_in2017_years",
    "w1_q34",

    # how couples met
    "w1_q24_met_online",
    "w1_q24_met_through_friend",
    "w1_q24_met_through_family",
    "w1_q24_met_as_through_cowork",
    "w1_q24_school",
    "w1_q24_college",
    "w1_q24_church",
    "w1_q24_bar_restaurant",
    "w1_q24_party",
    "w1_q32",

    # wave 3 outcomes
    "w3_partner_type",
    "w3_live_w_partner",
    "w3_relationship_end_combo",
    "w3_partner_source",
    "w3_breakup_source",
    "w3_relationship_duration_yrs"
]

existing_candidate_vars = [c for c in candidate_vars if c in df.columns]
missing_candidate_vars = [c for c in candidate_vars if c not in df.columns]

print("Existing candidate vars:", len(existing_candidate_vars))
print("Missing candidate vars:", missing_candidate_vars)

Existing candidate vars: 35
Missing candidate vars: []


## Creating reduced working dataset

This file keeps only the variables selected for the project, making later notebooks easier to manage and faster to run.

In [6]:
df_small = df[existing_candidate_vars].copy()
print(df_small.shape)
df_small.head()

(3510, 35)


,caseid_new,w1_weight_combo,w2_attrition_adj_weights,w3_attrition_adj_weight,w2_surveyed,w3_surveyed,w1_ppage,w1_ppeduc,w1_ppincimp,w1_ppgender,...,w1_q24_church,w1_q24_bar_restaurant,w1_q24_party,w1_q32,w3_partner_type,w3_live_w_partner,w3_relationship_end_combo,w3_partner_source,w3_breakup_source,w3_relationship_duration_yrs
0,53001.0,0.426861,0.380351,0.400185,yes,yes,48.0,HIGH SCHOOL GRADUATE - high school DIPLOMA or ...,"$60,000 to $74,999",Female,...,no,yes,no,"No, I did NOT meet [Partner Name] through the ...",in unmarried partnership,yes,no report of breakup or partner death,2020 wave,NaN,1.500000
1,71609.0,1.295508,0.953948,0.879258,yes,yes,68.0,"Some college, no degree","$50,000 to $59,999",Female,...,no,no,yes,"No, I did NOT meet [Partner Name] through the ...",married,yes,no report of breakup or partner death,2017 wave,NaN,57.416668
2,106983.0,1.126573,0.724682,0.706467,yes,yes,39.0,Associate degree,"$85,000 to $99,999",Male,...,no,no,no,"No, I did NOT meet [Partner Name] through the ...",married,yes,no report of breakup or partner death,2017 wave,NaN,22.333334
3,121759.0,0.933440,0.793093,NaN,yes,no,54.0,HIGH SCHOOL GRADUATE - high school DIPLOMA or ...,"$100,000 to $124,999",Male,...,yes,no,no,"No, I did NOT meet [Partner Name] through the ...",not in wave 3,NaN,NaN,NaN,NaN,NaN
4,158083.0,0.931291,0.735473,0.655467,yes,yes,48.0,"Some college, no degree","$75,000 to $84,999",Male,...,no,no,no,"No, I did NOT meet [Partner Name] through the ...",unpartnered,NaN,no report of breakup or partner death,NaN,NaN,NaN


In [7]:
df_small.to_csv(DATA_PROCESSED / "hcmst_selected_variables.csv", index=False)
print("Saved:", DATA_PROCESSED / "hcmst_selected_variables.csv")

Saved: ../data/processed/hcmst_selected_variables.csv


## Defining Wave 1 active-relationship sample

For clustering and the main baseline relationship analysis, we keep only respondents who were in an active relationship in Wave 1:
- married
- partnered, not married

We exclude:
- unpartnered, has had past partner
- never had a partner

In [8]:
active_relationship_status = ["married", "partnered, not married"]

df_rel = df_small[
    df_small["w1_partnership_status"].astype(str).isin(active_relationship_status)
].copy()

print(df_rel.shape)
print(df_rel["w1_partnership_status"].value_counts(dropna=False))

(2862, 35)
w1_partnership_status
married                              2085
partnered, not married                777
never had a partner                     0
unpartnered, has had past partner       0
Name: count, dtype: int64


In [9]:
df_rel.to_csv(DATA_PROCESSED / "hcmst_wave1_relationship_sample.csv", index=False)
print("Saved:", DATA_PROCESSED / "hcmst_wave1_relationship_sample.csv")

Saved: ../data/processed/hcmst_wave1_relationship_sample.csv


In [10]:
missing_rel = (
    df_rel.isna().mean().sort_values(ascending=False) * 100
).round(2)

missing_rel

w3_breakup_source                  96.82
w3_relationship_duration_yrs       56.57
w3_live_w_partner                  55.31
w3_partner_source                  55.31
w3_attrition_adj_weight            51.19
w3_relationship_end_combo          51.19
w2_attrition_adj_weights           40.04
w1_relate_duration_in2017_years     3.39
w1_q24_met_as_through_cowork        2.66
w1_q24_met_through_family           2.66
w1_q24_met_through_friend           2.66
w1_q24_church                       2.66
w1_q24_college                      2.66
w1_q24_bar_restaurant               2.66
w1_q24_party                        2.66
w1_q24_school                       2.66
w1_q34                              0.52
w1_q19                              0.35
w1_q24_met_online                   0.21
w1_q32                              0.21
w1_same_sex_couple                  0.21
w1_married                          0.21
caseid_new                          0.00
w1_weight_combo                     0.00
w1_ppreg9       

In [11]:
cluster_candidate_vars = [
    "w1_ppage",
    "w1_ppgender",
    "w1_ppethm",
    "w1_ppwork",
    "w1_married",
    "w1_q19",
    "w1_same_sex_couple",
    "w1_relate_duration_in2017_years",
    "w1_q34",
    "w1_q24_met_online"
]

(
    df_rel[cluster_candidate_vars]
    .isna()
    .mean()
    .sort_values(ascending=False) * 100
).round(2)

w1_relate_duration_in2017_years    3.39
w1_q34                             0.52
w1_q19                             0.35
w1_q24_met_online                  0.21
w1_same_sex_couple                 0.21
w1_married                         0.21
w1_ppgender                        0.00
w1_ppage                           0.00
w1_ppethm                          0.00
w1_ppwork                          0.00
dtype: float64

## Creating working cleaned dataset

At this point, the Wave 1 active-relationship sample has been defined.

The next steps are:

- create a working copy for cleaning
- recode variables into analysis-friendly formats
- convert numeric-looking variables safely
- handle missing values
- create a clean model-ready baseline file
- save cleaned outputs for later notebooks

In [12]:
df_clean = df_rel.copy()

print(df_clean.shape)
df_clean.head()

(2862, 35)


,caseid_new,w1_weight_combo,w2_attrition_adj_weights,w3_attrition_adj_weight,w2_surveyed,w3_surveyed,w1_ppage,w1_ppeduc,w1_ppincimp,w1_ppgender,...,w1_q24_church,w1_q24_bar_restaurant,w1_q24_party,w1_q32,w3_partner_type,w3_live_w_partner,w3_relationship_end_combo,w3_partner_source,w3_breakup_source,w3_relationship_duration_yrs
0,53001.0,0.426861,0.380351,0.400185,yes,yes,48.0,HIGH SCHOOL GRADUATE - high school DIPLOMA or ...,"$60,000 to $74,999",Female,...,no,yes,no,"No, I did NOT meet [Partner Name] through the ...",in unmarried partnership,yes,no report of breakup or partner death,2020 wave,NaN,1.500000
1,71609.0,1.295508,0.953948,0.879258,yes,yes,68.0,"Some college, no degree","$50,000 to $59,999",Female,...,no,no,yes,"No, I did NOT meet [Partner Name] through the ...",married,yes,no report of breakup or partner death,2017 wave,NaN,57.416668
2,106983.0,1.126573,0.724682,0.706467,yes,yes,39.0,Associate degree,"$85,000 to $99,999",Male,...,no,no,no,"No, I did NOT meet [Partner Name] through the ...",married,yes,no report of breakup or partner death,2017 wave,NaN,22.333334
3,121759.0,0.933440,0.793093,NaN,yes,no,54.0,HIGH SCHOOL GRADUATE - high school DIPLOMA or ...,"$100,000 to $124,999",Male,...,yes,no,no,"No, I did NOT meet [Partner Name] through the ...",not in wave 3,NaN,NaN,NaN,NaN,NaN
5,164061.0,0.931291,0.605404,0.342626,yes,yes,59.0,"Some college, no degree","$85,000 to $99,999",Male,...,no,no,no,"No, I did NOT meet [Partner Name] through the ...",married,yes,no report of breakup or partner death,2017 wave,NaN,28.250000


## Recoding analysis variables

The HCMST public file includes a mix of SPSS-style categorical variables and numeric-looking values stored as labels.

To prepare the data for later clustering and modeling, we recode several variables:

- yes/no variables into 1/0
- gender into numeric form
- relationship quality into an ordered score
- numeric-looking variables into proper numeric dtype

In [13]:
binary_cols = [
    "w1_married",
    "w1_q24_met_online",
    "w2_surveyed",
    "w3_surveyed"
]

for col in binary_cols:
    if col in df_clean.columns:
        df_clean[col] = recode_yes_no(df_clean[col])

df_clean[binary_cols].head()

,w1_married,w1_q24_met_online,w2_surveyed,w3_surveyed
0,1.0,0.0,1,1
1,1.0,0.0,1,1
2,1.0,0.0,1,1
3,1.0,0.0,1,0
5,1.0,0.0,1,1


In [14]:
df_clean["w1_same_sex_couple"] = recode_same_sex(df_clean["w1_same_sex_couple"])

df_clean["w1_same_sex_couple"].value_counts(dropna=False)

w1_same_sex_couple
NaN    2644
1.0     218
Name: count, dtype: int64

In [15]:
df_clean["w1_ppgender"] = (
    df_clean["w1_ppgender"]
    .astype(str)
    .str.strip()
)

df_clean["w1_ppgender_num"] = df_clean["w1_ppgender"].map({
    "Male": 0,
    "Female": 1
})

df_clean[["w1_ppgender", "w1_ppgender_num"]].head()

,w1_ppgender,w1_ppgender_num
0,Female,1
1,Female,1
2,Male,0
3,Male,0
5,Male,0


In [17]:
df_clean["w1_q34_score"] = recode_quality(df_clean["w1_q34"])

df_clean[["w1_q34", "w1_q34_score"]].head(10)

,w1_q34,w1_q34_score
0,Excellent,5.0
1,Excellent,5.0
2,Excellent,5.0
3,Excellent,5.0
5,Excellent,5.0
7,Good,4.0
8,Excellent,5.0
9,Good,4.0
10,Fair,3.0
11,Good,4.0


## Converting numeric-like variables

Some variables that should behave like numbers may still be stored as categorical or object types after import.

We convert the main numeric variables safely using `pd.to_numeric(..., errors="coerce")`.

In [18]:
numeric_cols = [
    "w1_ppage",
    "w1_relate_duration_in2017_years",
    "w3_relationship_duration_yrs",
    "w1_weight_combo",
    "w2_attrition_adj_weights",
    "w3_attrition_adj_weight"
]

for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = safe_numeric(df_clean[col])

df_clean[numeric_cols].dtypes

w1_ppage                           float64
w1_relate_duration_in2017_years    float64
w3_relationship_duration_yrs       float64
w1_weight_combo                    float64
w2_attrition_adj_weights           float64
w3_attrition_adj_weight            float64
dtype: object

In [19]:
df_clean[numeric_cols].describe()

,w1_ppage,w1_relate_duration_in2017_years,w3_relationship_duration_yrs,w1_weight_combo,w2_attrition_adj_weights,w3_attrition_adj_weight
count,2862.000000,2765.000000,1243.000000,2862.000000,1716.000000,1397.000000
mean,49.369322,21.553466,28.021588,0.994854,0.981322,0.978445
std,16.241715,16.650276,16.654410,0.421139,0.794983,1.036929
min,18.000000,-0.333333,0.083333,0.058970,0.025204,0.026903
25%,35.000000,7.250000,13.958333,0.819488,0.518798,0.450199
50%,51.000000,18.083334,26.583334,1.005959,0.766728,0.703767
75%,62.000000,34.166668,40.791666,1.230292,1.184025,1.139459
max,93.000000,75.166664,74.666664,3.002951,7.077054,14.638303


## Handling missing values

Missingness is low for the main Wave 1 baseline variables used later in analysis.

We use simple, transparent imputation rules:

- numeric variables → median
- binary variables → mode
- recoded quality score → median
- gender numeric → mode
- remaining text variables → "Missing"

This preserves sample size while keeping the cleaning logic easy to explain.

In [20]:
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

In [21]:
binary_impute_cols = [
    "w1_married",
    "w1_q24_met_online",
    "w2_surveyed",
    "w3_surveyed",
    "w1_same_sex_couple"
]

for col in binary_impute_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

In [22]:
df_clean["w1_q34_score"] = df_clean["w1_q34_score"].fillna(
    df_clean["w1_q34_score"].median()
)

df_clean["w1_ppgender_num"] = df_clean["w1_ppgender_num"].fillna(
    df_clean["w1_ppgender_num"].mode()[0]
)

In [23]:
text_cols = df_clean.select_dtypes(include=["object"]).columns

for col in text_cols:
    df_clean[col] = df_clean[col].fillna("Missing")

/tmp/ipykernel_123763/1157473273.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df_clean.select_dtypes(include=["object"]).columns


In [24]:
(df_clean.isna().mean() * 100).sort_values(ascending=False).head(20)

w3_breakup_source               96.820405
w3_live_w_partner               55.310971
w3_partner_source               55.310971
w3_relationship_end_combo       51.187980
w1_q24_met_through_friend        2.655486
w1_q24_college                   2.655486
w1_q24_church                    2.655486
w1_q24_party                     2.655486
w1_q24_bar_restaurant            2.655486
w1_q24_met_as_through_cowork     2.655486
w1_q24_school                    2.655486
w1_q24_met_through_family        2.655486
w1_q34                           0.524109
w1_q19                           0.349406
w1_q32                           0.209644
w2_surveyed                      0.000000
w1_weight_combo                  0.000000
caseid_new                       0.000000
w3_attrition_adj_weight          0.000000
w2_attrition_adj_weights         0.000000
dtype: float64

## Creating model-ready baseline dataset

This file is not yet the final clustering matrix.

Instead, it is a compact cleaned baseline dataset containing:

- respondent ID
- Wave 1 weight
- core demographics
- core relationship structure variables
- meeting context
- cleaned numeric versions where needed

This will be the bridge between cleaning and later analysis notebooks.

In [25]:
model_vars = [
    "caseid_new",
    "w1_weight_combo",

    "w1_ppage",
    "w1_ppgender",
    "w1_ppgender_num",
    "w1_ppethm",
    "w1_ppwork",

    "w1_married",
    "w1_q19",
    "w1_same_sex_couple",

    "w1_relate_duration_in2017_years",
    "w1_q34",
    "w1_q34_score",

    "w1_q24_met_online"
]

df_model = df_clean[model_vars].copy()

print(df_model.shape)
df_model.head()

(2862, 14)


,caseid_new,w1_weight_combo,w1_ppage,w1_ppgender,w1_ppgender_num,w1_ppethm,w1_ppwork,w1_married,w1_q19,w1_same_sex_couple,w1_relate_duration_in2017_years,w1_q34,w1_q34_score,w1_q24_met_online
0,53001.0,0.426861,48.0,Female,1,"2+ Races, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,3.583333,Excellent,5.0,0.0
1,71609.0,1.295508,68.0,Female,1,"White, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,52.750000,Excellent,5.0,0.0
2,106983.0,1.126573,39.0,Male,0,"White, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,17.583334,Excellent,5.0,0.0
3,121759.0,0.933440,54.0,Male,0,"White, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,27.416666,Excellent,5.0,0.0
5,164061.0,0.931291,59.0,Male,0,"White, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,23.583334,Excellent,5.0,0.0



### These variables provide a strong baseline description of respondents’ relationship profiles in Wave 1:

- demographics: age, gender, ethnicity, work
- relationship structure: married, cohabiting, same-sex couple, duration
- relationship quality: ordered score from poor to excellent
- meeting context: met online or not
- weight: Wave 1 survey weight

This file is suitable as a clean starting point for later clustering and modeling decisions.

##  Before saving the cleaned datasets, we inspect the structure and verify that the cleaned baseline file looks correct.

In [26]:
df_model.info()

<class 'pandas.DataFrame'>
Index: 2862 entries, 0 to 3509
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype   
---  ------                           --------------  -----   
 0   caseid_new                       2862 non-null   float64 
 1   w1_weight_combo                  2862 non-null   float64 
 2   w1_ppage                         2862 non-null   float64 
 3   w1_ppgender                      2862 non-null   str     
 4   w1_ppgender_num                  2862 non-null   int64   
 5   w1_ppethm                        2862 non-null   category
 6   w1_ppwork                        2862 non-null   category
 7   w1_married                       2862 non-null   float64 
 8   w1_q19                           2852 non-null   category
 9   w1_same_sex_couple               2862 non-null   float64 
 10  w1_relate_duration_in2017_years  2862 non-null   float64 
 11  w1_q34                           2847 non-null   category
 12  w1_q34_score          

In [27]:
df_model.isna().sum()

caseid_new                          0
w1_weight_combo                     0
w1_ppage                            0
w1_ppgender                         0
w1_ppgender_num                     0
w1_ppethm                           0
w1_ppwork                           0
w1_married                          0
w1_q19                             10
w1_same_sex_couple                  0
w1_relate_duration_in2017_years     0
w1_q34                             15
w1_q34_score                        0
w1_q24_met_online                   0
dtype: int64

In [28]:
df_model.head(10)

,caseid_new,w1_weight_combo,w1_ppage,w1_ppgender,w1_ppgender_num,w1_ppethm,w1_ppwork,w1_married,w1_q19,w1_same_sex_couple,w1_relate_duration_in2017_years,w1_q34,w1_q34_score,w1_q24_met_online
0,53001.0,0.426861,48.0,Female,1,"2+ Races, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,3.583333,Excellent,5.0,0.0
1,71609.0,1.295508,68.0,Female,1,"White, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,52.750000,Excellent,5.0,0.0
2,106983.0,1.126573,39.0,Male,0,"White, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,17.583334,Excellent,5.0,0.0
3,121759.0,0.933440,54.0,Male,0,"White, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,27.416666,Excellent,5.0,0.0
5,164061.0,0.931291,59.0,Male,0,"White, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,23.583334,Excellent,5.0,0.0
7,212249.0,1.347071,55.0,Female,1,"Black, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,34.250000,Good,4.0,0.0
8,214227.0,0.975956,73.0,Female,1,"White, Non-Hispanic",Not working - retired,1.0,Yes,1.0,23.666666,Excellent,5.0,0.0
9,218351.0,0.966458,46.0,Male,0,"White, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,21.000000,Good,4.0,0.0
10,220655.0,0.456147,43.0,Male,0,"2+ Races, Non-Hispanic",Working - as a paid employee,1.0,Yes,1.0,16.416666,Fair,3.0,0.0
11,291177.0,1.131096,57.0,Female,1,"White, Non-Hispanic",Working - as a paid employee,0.0,No,1.0,3.416667,Good,4.0,0.0


## Adjusting final cleaning outputs for both analysis and machine learning

The notebook currently contains a compact cleaned baseline file, but for the rest of the project we need two separate outputs:

- `df_analysis`: readable dataset for EDA, cluster interpretation, and reporting
- `df_ml`: numeric / ML-ready dataset for clustering and predictive modeling

Before creating those outputs, we first correct the recoding of `w1_same_sex_couple`.

In [29]:
df_clean["w1_same_sex_couple_raw"] = df_rel["w1_same_sex_couple"].astype(str).str.strip()

df_clean["w1_same_sex_couple_num"] = df_clean["w1_same_sex_couple_raw"].map({
    "same_sex_couple": 1,
    "NOT same-sex souple": 0,
    "NOT same-sex couple": 0
})

df_clean["w1_same_sex_couple_num"].value_counts(dropna=False)

w1_same_sex_couple_num
0.0    2638
1.0     218
NaN       6
Name: count, dtype: int64

In [30]:
df_clean["w1_same_sex_couple_num"] = df_clean["w1_same_sex_couple_num"].fillna(
    df_clean["w1_same_sex_couple_num"].mode()[0]
)

df_clean["w1_same_sex_couple_num"].isna().sum()

np.int64(0)

## Create analysis-ready dataset

The analysis-ready dataset keeps readable categorical labels for interpretation and reporting.

This file is useful for:
- descriptive tables
- cluster profiling
- charts
- presentation slides
- writing conclusions in plain language

In [31]:
analysis_vars = [
    "caseid_new",
    "w1_weight_combo",

    "w1_ppage",
    "w1_ppgender",
    "w1_ppeduc",
    "w1_ppincimp",
    "w1_ppethm",
    "w1_ppwork",

    "w1_partnership_status",
    "w1_q19",
    "w1_married",
    "w1_same_sex_couple_raw",
    "w1_relate_duration_in2017_years",
    "w1_q34",
    "w1_q24_met_online",

    "w2_surveyed",
    "w3_surveyed",
    "w2_attrition_adj_weights",
    "w3_attrition_adj_weight",

    "w3_partner_type",
    "w3_live_w_partner",
    "w3_relationship_end_combo",
    "w3_partner_source",
    "w3_breakup_source",
    "w3_relationship_duration_yrs"
]

df_analysis = df_clean[analysis_vars].copy()
print(df_analysis.shape)
df_analysis.head()

(2862, 25)


,caseid_new,w1_weight_combo,w1_ppage,w1_ppgender,w1_ppeduc,w1_ppincimp,w1_ppethm,w1_ppwork,w1_partnership_status,w1_q19,...,w2_surveyed,w3_surveyed,w2_attrition_adj_weights,w3_attrition_adj_weight,w3_partner_type,w3_live_w_partner,w3_relationship_end_combo,w3_partner_source,w3_breakup_source,w3_relationship_duration_yrs
0,53001.0,0.426861,48.0,Female,HIGH SCHOOL GRADUATE - high school DIPLOMA or ...,"$60,000 to $74,999","2+ Races, Non-Hispanic",Working - as a paid employee,married,Yes,...,1,1,0.380351,0.400185,in unmarried partnership,yes,no report of breakup or partner death,2020 wave,NaN,1.500000
1,71609.0,1.295508,68.0,Female,"Some college, no degree","$50,000 to $59,999","White, Non-Hispanic",Working - as a paid employee,married,Yes,...,1,1,0.953948,0.879258,married,yes,no report of breakup or partner death,2017 wave,NaN,57.416668
2,106983.0,1.126573,39.0,Male,Associate degree,"$85,000 to $99,999","White, Non-Hispanic",Working - as a paid employee,married,Yes,...,1,1,0.724682,0.706467,married,yes,no report of breakup or partner death,2017 wave,NaN,22.333334
3,121759.0,0.933440,54.0,Male,HIGH SCHOOL GRADUATE - high school DIPLOMA or ...,"$100,000 to $124,999","White, Non-Hispanic",Working - as a paid employee,married,Yes,...,1,0,0.793093,0.703767,not in wave 3,NaN,NaN,NaN,NaN,26.583334
5,164061.0,0.931291,59.0,Male,"Some college, no degree","$85,000 to $99,999","White, Non-Hispanic",Working - as a paid employee,married,Yes,...,1,1,0.605404,0.342626,married,yes,no report of breakup or partner death,2017 wave,NaN,28.250000


In [32]:
analysis_text_cols = df_analysis.select_dtypes(include=["object", "string", "category"]).columns

for col in analysis_text_cols:
    df_analysis[col] = df_analysis[col].astype(str).replace("nan", "Missing")
    df_analysis[col] = df_analysis[col].fillna("Missing")

df_analysis.isna().sum().sort_values(ascending=False).head(20)

caseid_new                         0
w1_weight_combo                    0
w1_ppage                           0
w1_ppgender                        0
w1_ppeduc                          0
w1_ppincimp                        0
w1_ppethm                          0
w1_ppwork                          0
w1_partnership_status              0
w1_q19                             0
w1_married                         0
w1_same_sex_couple_raw             0
w1_relate_duration_in2017_years    0
w1_q34                             0
w1_q24_met_online                  0
w2_surveyed                        0
w3_surveyed                        0
w2_attrition_adj_weights           0
w3_attrition_adj_weight            0
w3_partner_type                    0
dtype: int64

## Create ML-ready dataset

The ML-ready dataset keeps only numeric or already-recoded features.

This file is intended for:
- clustering
- classification / regression
- scaling
- feature engineering
- model evaluation

In [33]:
ml_vars = [
    "caseid_new",
    "w1_weight_combo",

    "w1_ppage",
    "w1_ppgender_num",

    "w1_married",
    "w1_q24_met_online",
    "w2_surveyed",
    "w3_surveyed",

    "w1_same_sex_couple_num",

    "w1_relate_duration_in2017_years",
    "w1_q34_score",

    "w2_attrition_adj_weights",
    "w3_attrition_adj_weight"
]

df_ml = df_clean[ml_vars].copy()

print(df_ml.shape)
df_ml.head()

(2862, 13)


,caseid_new,w1_weight_combo,w1_ppage,w1_ppgender_num,w1_married,w1_q24_met_online,w2_surveyed,w3_surveyed,w1_same_sex_couple_num,w1_relate_duration_in2017_years,w1_q34_score,w2_attrition_adj_weights,w3_attrition_adj_weight
0,53001.0,0.426861,48.0,1,1.0,0.0,1,1,0.0,3.583333,5.0,0.380351,0.400185
1,71609.0,1.295508,68.0,1,1.0,0.0,1,1,0.0,52.750000,5.0,0.953948,0.879258
2,106983.0,1.126573,39.0,0,1.0,0.0,1,1,0.0,17.583334,5.0,0.724682,0.706467
3,121759.0,0.933440,54.0,0,1.0,0.0,1,0,0.0,27.416666,5.0,0.793093,0.703767
5,164061.0,0.931291,59.0,0,1.0,0.0,1,1,0.0,23.583334,5.0,0.605404,0.342626


In [34]:
df_ml.isna().sum()

caseid_new                         0
w1_weight_combo                    0
w1_ppage                           0
w1_ppgender_num                    0
w1_married                         0
w1_q24_met_online                  0
w2_surveyed                        0
w3_surveyed                        0
w1_same_sex_couple_num             0
w1_relate_duration_in2017_years    0
w1_q34_score                       0
w2_attrition_adj_weights           0
w3_attrition_adj_weight            0
dtype: int64

### Why these ML-ready variables are included

These variables describe respondents’ baseline relationship profile in Wave 1 and include the relevant survey weights:

- age
- gender
- married status
- same-sex couple status
- relationship quality
- met online
- relationship duration
- wave participation / attrition weights

This creates a compact baseline file suitable for later clustering and predictive modeling.

In [35]:
print("df_analysis shape:", df_analysis.shape)
print("df_ml shape:", df_ml.shape)

print("\nMissing values in df_analysis:")
print(df_analysis.isna().sum().sort_values(ascending=False).head(10))

print("\nMissing values in df_ml:")
print(df_ml.isna().sum().sort_values(ascending=False).head(10))

df_analysis shape: (2862, 25)
df_ml shape: (2862, 13)

Missing values in df_analysis:
caseid_new               0
w1_weight_combo          0
w1_ppage                 0
w1_ppgender              0
w1_ppeduc                0
w1_ppincimp              0
w1_ppethm                0
w1_ppwork                0
w1_partnership_status    0
w1_q19                   0
dtype: int64

Missing values in df_ml:
caseid_new                         0
w1_weight_combo                    0
w1_ppage                           0
w1_ppgender_num                    0
w1_married                         0
w1_q24_met_online                  0
w2_surveyed                        0
w3_surveyed                        0
w1_same_sex_couple_num             0
w1_relate_duration_in2017_years    0
dtype: int64


## Save final cleaned outputs

We now save:
- the broad cleaned Wave 1 relationship sample
- the analysis-ready dataset
- the ML-ready dataset

In [36]:
df_clean.to_csv(
    DATA_PROCESSED / "hcmst_wave1_cleaned.csv",
    index=False
)

df_analysis.to_csv(
    DATA_PROCESSED / "hcmst_wave1_analysis_ready.csv",
    index=False
)

df_ml.to_csv(
    DATA_PROCESSED / "hcmst_wave1_ml_ready.csv",
    index=False
)

print("Saved:", DATA_PROCESSED / "hcmst_wave1_cleaned.csv")
print("Saved:", DATA_PROCESSED / "hcmst_wave1_analysis_ready.csv")
print("Saved:", DATA_PROCESSED / "hcmst_wave1_ml_ready.csv")

Saved: ../data/processed/hcmst_wave1_cleaned.csv
Saved: ../data/processed/hcmst_wave1_analysis_ready.csv
Saved: ../data/processed/hcmst_wave1_ml_ready.csv


In [37]:
cleaning_summary = pd.DataFrame({
    "dataset": [
        "raw_full_dataset",
        "selected_variables_dataset",
        "wave1_relationship_sample",
        "wave1_cleaned",
        "wave1_analysis_ready",
        "wave1_ml_ready"
    ],
    "n_rows": [
        len(df),
        len(df_small),
        len(df_rel),
        len(df_clean),
        len(df_analysis),
        len(df_ml)
    ],
    "n_columns": [
        df.shape[1],
        df_small.shape[1],
        df_rel.shape[1],
        df_clean.shape[1],
        df_analysis.shape[1],
        df_ml.shape[1]
    ]
})

cleaning_summary

,dataset,n_rows,n_columns
0,raw_full_dataset,3510,725
1,selected_variables_dataset,3510,35
2,wave1_relationship_sample,2862,35
3,wave1_cleaned,2862,39
4,wave1_analysis_ready,2862,25
5,wave1_ml_ready,2862,13


In [38]:
cleaning_summary.to_csv(
    DATA_PROCESSED / "cleaning_summary.csv",
    index=False
)

print("Saved:", DATA_PROCESSED / "cleaning_summary.csv")

Saved: ../data/processed/cleaning_summary.csv


In [ ]:
df_analysis.isna().sum()


caseid_new                         0
w1_weight_combo                    0
w1_ppage                           0
w1_ppgender_num                    0
w1_married                         0
w1_q24_met_online                  0
w2_surveyed                        0
w3_surveyed                        0
w1_same_sex_couple_num             0
w1_relate_duration_in2017_years    0
w1_q34_score                       0
w2_attrition_adj_weights           0
w3_attrition_adj_weight            0
dtype: int64

In [42]:
df_ml.isna().sum()


caseid_new                         0
w1_weight_combo                    0
w1_ppage                           0
w1_ppgender_num                    0
w1_married                         0
w1_q24_met_online                  0
w2_surveyed                        0
w3_surveyed                        0
w1_same_sex_couple_num             0
w1_relate_duration_in2017_years    0
w1_q34_score                       0
w2_attrition_adj_weights           0
w3_attrition_adj_weight            0
dtype: int64

## Cleaning notebook outputs

This notebook created:

- `hcmst_selected_variables.csv`  
  reduced working dataset with selected variables only

- `hcmst_wave1_relationship_sample.csv`  
  active Wave 1 relationships only

- `hcmst_wave1_cleaned.csv`  
  cleaned Wave 1 relationship sample with recoded variables

- `hcmst_wave1_analysis_ready.csv`  
  readable cleaned dataset for interpretation, reporting, and cluster profiling

- `hcmst_wave1_ml_ready.csv`  
  numeric ML-ready dataset for clustering and supervised models

- `cleaning_summary.csv`  
  dataset size summary across cleaning stages